<a href="https://colab.research.google.com/github/omzeybek/Yeditepe_Data_Science_DATS501/blob/main/Segmentation_Synthetic_Data_Example.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# DATS 501 — Customer Segmentation with Synthetic Data

**Unsupervised Learning / Coding Session**

This notebook demonstrates **segmentation** (clustering customers into groups) using **synthetic data**.
We generate fake customer profiles, discover natural groups with K-Means, and interpret each segment.

## What is segmentation?

Segmentation divides a population into meaningful subgroups that share similar characteristics.
Common use cases include:

- Marketing: target promotions to high-value customers
- Retail: identify bargain hunters vs. premium buyers
- Banking: detect risk profiles among loan applicants

Because real customer data is often private, we start with **synthetic data** that mimics realistic patterns.

## 1. Import packages

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

np.random.seed(42)
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (8, 5)

## 2. Generate synthetic customer data

We simulate **600 customers** across **4 hidden segments**:

| Segment | Profile |
|---------|---------|
| 0 | Young, low income, moderate spending |
| 1 | Middle-aged, high income, high spending |
| 2 | Older, moderate income, low spending |
| 3 | Young, moderate income, very high spending |

In [ ]:
def generate_synthetic_customers(n_per_segment=150, random_state=42):
    """Create synthetic customer profiles with 4 underlying segments."""
    rng = np.random.default_rng(random_state)

    segment_params = [
        {"age": (22, 8), "income": (28_000, 6_000), "spending_score": (55, 12)},
        {"age": (45, 7), "income": (95_000, 12_000), "spending_score": (78, 10)},
        {"age": (62, 6), "income": (52_000, 8_000), "spending_score": (25, 8)},
        {"age": (28, 6), "income": (48_000, 7_000), "spending_score": (88, 9)},
    ]

    rows = []
    for segment_id, params in enumerate(segment_params):
        for _ in range(n_per_segment):
            age = max(18, rng.normal(*params["age"]))
            income = max(15_000, rng.normal(*params["income"]))
            spending = np.clip(rng.normal(*params["spending_score"]), 0, 100)
            visits = max(1, int(rng.poisson(spending / 25)))
            rows.append({
                "customer_id": f"C{len(rows) + 1:04d}",
                "age": round(age, 1),
                "annual_income": round(income, 2),
                "spending_score": round(spending, 1),
                "monthly_visits": visits,
                "true_segment": segment_id,  # hidden label for evaluation only
            })

    df = pd.DataFrame(rows)
    return df.sample(frac=1, random_state=random_state).reset_index(drop=True)


customers = generate_synthetic_customers()
customers.head(10)

## 3. Exploratory data analysis

In [ ]:
feature_cols = ["age", "annual_income", "spending_score", "monthly_visits"]

print(customers[feature_cols].describe().round(2))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.scatterplot(
    data=customers,
    x="annual_income",
    y="spending_score",
    hue="true_segment",
    palette="tab10",
    alpha=0.7,
    ax=axes[0],
)
axes[0].set_title("True segments (hidden in real projects)")
axes[0].set_xlabel("Annual income ($)")
axes[0].set_ylabel("Spending score (0-100)")

sns.scatterplot(
    data=customers,
    x="age",
    y="monthly_visits",
    hue="true_segment",
    palette="tab10",
    alpha=0.7,
    ax=axes[1],
)
axes[1].set_title("Age vs. monthly visits")

plt.tight_layout()
plt.show()

## 4. Prepare features for clustering

K-Means uses distance between points, so we **standardize** features to put them on a comparable scale.

In [ ]:
X = customers[feature_cols].copy()

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print("Scaled feature matrix shape:", X_scaled.shape)

## 5. Choose the number of segments (Elbow method)

We try different values of **k** and look for the point where adding more clusters gives diminishing returns.

In [ ]:
k_range = range(2, 11)
inertia_values = []
silhouette_values = []

for k in k_range:
    model = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = model.fit_predict(X_scaled)
    inertia_values.append(model.inertia_)
    silhouette_values.append(silhouette_score(X_scaled, labels))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(list(k_range), inertia_values, marker="o")
axes[0].set_title("Elbow method (inertia)")
axes[0].set_xlabel("Number of clusters (k)")
axes[0].set_ylabel("Inertia")

axes[1].plot(list(k_range), silhouette_values, marker="o", color="darkorange")
axes[1].set_title("Silhouette score by k")
axes[1].set_xlabel("Number of clusters (k)")
axes[1].set_ylabel("Silhouette score")

plt.tight_layout()
plt.show()

best_k = list(k_range)[int(np.argmax(silhouette_values))]
print(f"Best k by silhouette score: {best_k}")

## 6. Fit K-Means segmentation model

We use **k = 4** because our synthetic data was built with four underlying groups.

In [ ]:
N_CLUSTERS = 4

kmeans = KMeans(n_clusters=N_CLUSTERS, random_state=42, n_init=10)
customers["segment"] = kmeans.fit_predict(X_scaled)

print(f"Silhouette score: {silhouette_score(X_scaled, customers['segment']):.3f}")
print(customers["segment"].value_counts().sort_index())

## 7. Profile each segment

Average feature values help us give each cluster a business-friendly name.

In [ ]:
segment_profile = (
    customers.groupby("segment")[feature_cols]
    .mean()
    .round(2)
    .rename(columns={
        "age": "avg_age",
        "annual_income": "avg_income",
        "spending_score": "avg_spending",
        "monthly_visits": "avg_visits",
    })
)

segment_profile["size"] = customers["segment"].value_counts().sort_index().values
segment_profile

In [ ]:
segment_labels = {
    0: "Budget shoppers",
    1: "Premium loyalists",
    2: "Frugal seniors",
    3: "Trend spenders",
}

customers["segment_name"] = customers["segment"].map(segment_labels)

fig, ax = plt.subplots(figsize=(9, 6))
sns.scatterplot(
    data=customers,
    x="annual_income",
    y="spending_score",
    hue="segment_name",
    palette="Set2",
    alpha=0.75,
    ax=ax,
)
centers = scaler.inverse_transform(kmeans.cluster_centers_)
ax.scatter(
    centers[:, 1],
    centers[:, 2],
    s=250,
    c="black",
    marker="X",
    label="Cluster centers",
)
ax.set_title("Discovered customer segments")
ax.set_xlabel("Annual income ($)")
ax.set_ylabel("Spending score (0-100)")
ax.legend(bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.show()

## 8. Compare with true segments (optional evaluation)

In real projects we do not have ground-truth labels. Here we compare discovered clusters to the hidden segments used when generating the data.

In [ ]:
from sklearn.metrics import adjusted_rand_score

ari = adjusted_rand_score(customers["true_segment"], customers["segment"])
print(f"Adjusted Rand Index (1.0 = perfect match): {ari:.3f}")

cross_tab = pd.crosstab(
    customers["true_segment"],
    customers["segment"],
    rownames=["True segment"],
    colnames=["Predicted segment"],
)
cross_tab

## 9. Summary

In this notebook we:

1. Generated **synthetic customer data** with four hidden groups
2. Explored relationships between age, income, spending, and visits
3. Scaled features and used the **Elbow method** + **silhouette score** to pick k
4. Applied **K-Means** to discover customer segments
5. Profiled each segment for actionable business interpretation

### Next steps

- Try different numbers of clusters and compare segment profiles
- Add more features (e.g., product categories, region)
- Use **PCA** to visualize high-dimensional segments
- Apply segmentation to real (anonymized) retail or banking data